In [31]:
import itertools
import json
import os

import numpy as np
from pathlib import Path
import pandas as pd

In [32]:
BASE_DIR = "../../evaluation/output_evals/emotiontalk"
DIRECTION_PAIRS = ['zh_en']
SYSTEM_NAMES = [
    'whisper',
    'seamlessm4t',
    'canary-v2',
    'owsm4.0-ctc',
    'aya_whisper',
    'gemma_whisper',
    'tower_whisper',
    'aya_seamlessm4t',
    'gemma_seamlessm4t',
    'tower_seamlessm4t',
    'aya_canary-v2',
    'gemma_canary-v2',
    'tower_canary-v2',
    'aya_owsm4.0-ctc',
    'gemma_owsm4.0-ctc',
    'tower_owsm4.0-ctc',
    'desta2-8b',
    'qwen2audio-7b',
    'phi4multimodal',
    'voxtral-small-24b',
]

In [33]:
def load_results_summaries(base_dir, direction_pairs, system_names):
    """
    Loads all result summaries from a directory structure.

    Args:
        base_dir (str or Path): The base directory for the evaluation outputs.
        direction_pairs (list): A list of language direction strings (e.g., 'en_de').
        system_names (list): A list of system name strings.

    Returns:
        dict: A nested dictionary containing the loaded data, structured as
              {direction: {system: [results]}}.
    """
    base_path = Path(base_dir)
    all_results = {}

    # Use itertools.product to cleanly iterate over all combinations
    for direction, system in itertools.product(direction_pairs, system_names):
        summary_path = base_path / system / direction / 'results.jsonl'
        
        # Initialize the nested dictionary structure
        if direction not in all_results:
            all_results[direction] = {}

        try:
            with summary_path.open('r', encoding='utf-8') as f:
                all_results[direction][system] = [json.loads(line) for line in f]
                
        except FileNotFoundError:
            print(f"Warning: File not found, skipping: {summary_path}")
            all_results[direction][system] = None # Or [] if you prefer an empty list
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON in {summary_path}: {e}")
            all_results[direction][system] = None

    return all_results

In [34]:
def convert_results_to_dataframe(results_data):
    """
    Converts the nested dictionary of results into a single pandas DataFrame.

    Each row corresponds to a single entry, with 'direction' and 'system'
    columns added, and all 'metrics' unpacked into separate columns.
    """
    all_records = []
    for direction, systems in results_data.items():
        for system, records in systems.items():
            if records is None:
                continue
            for record in records:
                # Separate metrics from the record
                metrics = record.pop("metrics", {})  # safely get metrics
                # Merge everything into one flat dict
                flat_record = {
                    "direction": direction,
                    "system": system,
                    **record,
                    **metrics,  # unpack metrics into top-level keys
                }
                all_records.append(flat_record)

    if not all_records:
        print("No records were found to create a DataFrame.")
        return pd.DataFrame()

    df = pd.DataFrame(all_records)

    # Put identifying info up front
    original_cols = [c for c in df.columns if c not in ["direction", "system"]]
    df = df[["direction", "system"] + original_cols]

    return df

In [35]:
def add_emotion_column(df, manifests_dir="../../manifests/emotiontalk"):
    """
    Adds an 'emotion' column to the DataFrame by reading all .jsonl files in the given directory.
    
    Args:
        df (pd.DataFrame): The DataFrame containing at least a 'sample_id' column.
        manifests_dir (str or Path): Directory containing .jsonl manifest files.

    Returns:
        pd.DataFrame: The original DataFrame with a new 'emotion' column.
    """
    manifests_dir = Path(manifests_dir)
    emotion_map = {}

    # Read all .jsonl files in the directory
    for file in manifests_dir.glob("*.jsonl"):
        with open(file, "r", encoding="utf-8") as f:
            for line in f:
                try:
                    record = json.loads(line)
                    sid = str(record.get("sample_id"))  # keep as string for safety
                    emo = record.get("benchmark_metadata", {}).get("emotion")
                    if sid and emo:
                        emotion_map[sid] = emo
                except json.JSONDecodeError:
                    continue  # skip bad lines just in case

    if not emotion_map:
        print("No emotion data found in manifest files.")
        df["emotion"] = None
        return df

    # Map the emotion values onto the DataFrame
    df = df.copy()
    df["emotion"] = df["sample_id"].astype(str).map(emotion_map)

    return df

In [36]:
def compute_emotion_strict_scores(df):
    """
    Computes mean metric scores and strict scores grouped by (system, emotion).
    
    Expects columns:
      - system
      - emotion
      - xcomet_qe_score
      - metricx_qe_score
      - linguapy_score (list/tuple of [flag, lang])
    """
    df = df.copy()

    # --- Split linguapy_score into two separate columns ---
    df[["linguapy_flag", "linguapy_lang"]] = pd.DataFrame(
        df["linguapy_score"].tolist(), index=df.index
    )

    # --- Define penalties ---
    penalty_by_metric = {
        "metricx_qe": 25,
        "xcomet_qe": 0,
    }

    # --- Strict score per row ---
    for metric in penalty_by_metric.keys():
        df[f"{metric}_strict"] = df.apply(
            lambda row: row[f"{metric}_score"]
            if row["linguapy_flag"] == 0
            else penalty_by_metric[metric],
            axis=1,
        )

    # --- Aggregate by system × emotion ---
    agg_cols = {
        "linguapy_flag": "mean",  # average from 0–1
    }
    for metric in penalty_by_metric.keys():
        agg_cols[f"{metric}_score"] = "mean"
        agg_cols[f"{metric}_strict"] = "mean"

    result = (
        df.groupby(["system", "emotion"])
        .agg(agg_cols)
        .reset_index()
        .rename(columns={"linguapy_flag": "linguapy_avg"})
    )

    result['linguapy_avg'] = result['linguapy_avg']*100
    return result

In [37]:
results_full = load_results_summaries(BASE_DIR, DIRECTION_PAIRS, SYSTEM_NAMES)

In [38]:
df = convert_results_to_dataframe(results_full)

In [39]:
#Need to add the column for emotion ID
df = add_emotion_column(df)

In [40]:
col_map = {
    "linguapy_avg": "LinguaPy",
    "metricx_qe_strict": "QEMetricX_24-Strict-linguapy",
    "xcomet_qe_strict": "XCOMET-QE-Strict-linguapy"
}

# Collapse and get the metrics balanced by the linguapy score
metric_cols = ["linguapy_avg", "metricx_qe_strict", "xcomet_qe_strict", "metricx_qe_score", "xcomet_qe_score"]

for pair in DIRECTION_PAIRS:
    sub_df = df[df['direction']==pair]
    sub_df = compute_emotion_strict_scores(sub_df)
    sub_df["metricx_qe_strict"] = 100 - 4 * sub_df["metricx_qe_strict"]
    sub_df["metricx_qe_score"] = 100 - 4 * sub_df["metricx_qe_score"]

    emotion_counts = df[df['system'] == SYSTEM_NAMES[0]]['emotion'].value_counts(dropna=False)

    # Pivot so metrics are the top-level columns and emotions are a sublevel
    sub_df = sub_df.pivot(index="system", columns="emotion", values=metric_cols)
    # Rename the metric top-level column labels using col_map
    sub_df.columns = pd.MultiIndex.from_tuples([
        (col_map.get(top, top), emo) for top, emo in sub_df.columns
    ])

    # For each top-level metric, compute weighted average across emotion subcolumns
    top_metrics = sub_df.columns.get_level_values(0).unique()
    for top in top_metrics:
        # emotions present under this top-level metric
        emo_cols = sub_df[top].columns
        weights = emotion_counts.reindex(emo_cols).fillna(0)
        weighted = sub_df[top].multiply(weights, axis=1).sum(axis=1) / emotion_counts.sum()
        sub_df[(top, 'ALL')] = weighted
    # Put 'ALL' first per metric
    sub_df = sub_df.reindex(columns=pd.MultiIndex.from_tuples(sorted(sub_df.columns, key=lambda x: (x[0], x[1] != 'ALL', x[1]))))

    # Sort the index to match the order in SYSTEM_NAMES (keep only systems present)
    ordered_systems = [s for s in SYSTEM_NAMES if s in sub_df.index]
    if ordered_systems:
        sub_df = sub_df.reindex(index=ordered_systems)

    sub_df.to_csv(f"emotiontalk_{pair}.csv")

In [41]:
sub_df

LinguaPy                                              \
                         ALL      angry  disgusted    fearful      happy   
system                                                                     
whisper             6.687403   2.898551   2.189781  11.235955   9.604520   
seamlessm4t         2.643857   0.579710   1.459854   3.370787   5.084746   
canary-v2          26.231208  22.898551  16.058394  39.325843  33.898305   
owsm4.0-ctc        20.062208  13.913043  11.678832  32.584270  24.293785   
aya_whisper         3.058580   0.869565   0.000000   4.494382   5.084746   
gemma_whisper       4.147227   0.869565   0.000000   5.617978   5.084746   
tower_whisper       4.561949   1.159420   0.729927   6.741573   5.649718   
aya_seamlessm4t     3.006739   0.869565   0.000000   4.494382   3.389831   
gemma_seamlessm4t   4.199067   1.449275   0.729927   5.617978   5.084746   
tower_seamlessm4t   4.561949   0.869565   0.729927  10.112360   6.214689   
aya_canary-v2       6.480041   6.956522  10.218978   3.370787   5.084746   
gemma_canary-v2    78.745464  78.840580  78.832117  80.898876  76.271186   
tower_canary-v2    24.572317  27.536232  32.116788  26.966292  25.988701   
aya_owsm4.0-ctc     2.643857   0.579710   0.000000   2.247191   3.954802   
gemma_owsm4.0-ctc   4.147227   0.869565   0.000000   4.494382   6.779661   
tower_owsm4.0-ctc   3.576983   0.000000   1.459854   4.494382   5.084746   
desta2-8b           3.732504   0.579710   2.189781   3.370787   3.389831   
qwen2audio-7b       8.346293   6.086957   2.919708   5.617978  11.864407   
phi4multimodal     11.093831   7.536232   8.029197   8.988764  19.209040   
voxtral-small-24b   5.598756   1.449275   0.729927  11.235955   8.474576   

                                                    \
                     neutral        sad  surprised   
system                                               
whisper             7.203390   4.255319  11.888112   
seamlessm4t         2.542373   0.000000   7.692308   
canary-v2          26.165254  18.085106  32.167832   
owsm4.0-ctc        21.292373  11.702128  27.272727   
aya_whisper         3.813559   0.000000   4.895105   
gemma_whisper       5.614407   2.127660   5.594406   
tower_whisper       6.038136   2.127660   5.594406   
aya_seamlessm4t     3.813559   0.000000   6.293706   
gemma_seamlessm4t   5.508475   2.127660   4.895105   
tower_seamlessm4t   5.614407   2.127660   6.293706   
aya_canary-v2       5.932203   3.191489  11.188811   
gemma_canary-v2    79.237288  65.957447  85.314685   
tower_canary-v2    20.974576  24.468085  30.769231   
aya_owsm4.0-ctc     3.707627   1.063830   2.797203   
gemma_owsm4.0-ctc   5.826271   1.063830   3.496503   
tower_owsm4.0-ctc   4.872881   1.063830   4.895105   
desta2-8b           4.872881   3.191489   6.293706   
qwen2audio-7b      10.063559   5.319149   6.993007   
phi4multimodal     11.970339   6.382979  11.188811   
voxtral-small-24b   6.567797   2.127660   9.090909   

                  QEMetricX_24-Strict-linguapy             ...  \
                                           ALL      angry  ...   
system                                                     ...   
whisper                              75.166791  75.826612  ...   
seamlessm4t                          73.980930  70.907135  ...   
canary-v2                            27.965332  27.402494  ...   
owsm4.0-ctc                          32.872233  31.260474  ...   
aya_whisper                          84.714875  85.551115  ...   
gemma_whisper                        83.122857  84.729581  ...   
tower_whisper                        82.839326  84.601843  ...   
aya_seamlessm4t                      84.576096  85.196113  ...   
gemma_seamlessm4t                    82.360925  84.005115  ...   
tower_seamlessm4t                    82.578566  85.082675  ...   
aya_canary-v2                        63.658522  60.376047  ...   
gemma_canary-v2                       8.890545   9.404981  ...   
tower_canary-v2                      5